# 데이터 품질 도구 - 어노테이션 클래스 검증 + 박스 타이트닝

유영관님이 채팅에서 정리해준 두 가지를 도구로 만들었다.
1. annotation 클래스가 실제 이미지와 맞는지 확인하는 도구
2. 박스가 루즈하게 잡혀있어서(threshold 0.75 넘어가면 점수로 안 잡힘) 타이트하게 잡아주는 도구

둘 다 학습 없이 돌아가는 순수 전처리/검증 도구라 로컬 CPU에서도 바로 실행된다.

## STEP 0 : 환경설정

In [ ]:
#@title (1) Import Module
import os
import re
import json
import glob
import random
from collections import defaultdict

import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

random.seed(42)
np.random.seed(42)


In [ ]:
#@title (2) 데이터 경로 설정 (로컬 병합 데이터 우선, 없으면 kagglehub)
def _find_data_root():
    candidates = [
        "/Users/codeit/Downloads/sprint_ai_project1_data",
        "/content/sprint_ai_project1_data",
    ]
    for c in candidates:
        if os.path.isdir(os.path.join(c, "train_images")):
            return c
    import kagglehub
    kagglehub.login()
    path = kagglehub.competition_download("ai14-level-project")
    return os.path.join(path, "sprint_ai_project1_data")


DATA_ROOT = _find_data_root()
TRAIN_IMAGE_DIR = os.path.join(DATA_ROOT, "train_images")
TRAIN_ANNOTATION_DIR = os.path.join(DATA_ROOT, "train_annotations")

image_paths = glob.glob(os.path.join(TRAIN_IMAGE_DIR, "*.png"))
annotation_paths = glob.glob(os.path.join(TRAIN_ANNOTATION_DIR, "**", "*.json"), recursive=True)

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"이미지 {len(image_paths)}장, annotation {len(annotation_paths)}개")


In [ ]:
#@title (3) 이미지 단위 Annotation 병합 (dl_mapping_code 기준)
# 폴더명(K-000123 형태)이 진짜 약 코드이고, dl_mapping_code가 여기에 100% 일치한다는 걸
# 이미 전수 검증했다 (RetinaNet_ResNet50_FPN_v2_AIHub_Merged.ipynb와 동일한 기준).
# dl_idx는 5.5%만 일치해서 쓰면 안 된다.
def parse_dl_mapping_code(code):
    match = re.fullmatch(r"K-(\d+)", str(code).strip())
    if match is None:
        raise ValueError(f"예상하지 못한 dl_mapping_code 형식: {code!r}")
    return int(match.group(1))


image_stem_to_path = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
stem_to_ann_paths = defaultdict(list)
for p in annotation_paths:
    stem = os.path.splitext(os.path.basename(p))[0]
    stem_to_ann_paths[stem].append(p)

valid_stems = [s for s in stem_to_ann_paths if s in image_stem_to_path]


def build_image_records(valid_stems, stem_to_ann_paths):
    image_records = {}
    category_id_to_name = {}

    for stem in valid_stems:
        img_info = None
        annotations = []
        for ann_path in stem_to_ann_paths[stem]:
            with open(ann_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            file_img = data["images"][0]
            drug_id = parse_dl_mapping_code(file_img["dl_mapping_code"])
            category_id_to_name[drug_id] = file_img["dl_name"]
            if img_info is None:
                img_info = file_img
            for ann in data.get("annotations", []):
                if len(ann.get("bbox", [])) != 4:
                    continue
                annotations.append({
                    "bbox": ann["bbox"],
                    "category_id": drug_id,
                    "annotation_path": ann_path,  # 검증 도구에서 폴더코드 대조용으로 씀
                })
        image_records[stem] = {
            "file_name": img_info["file_name"],
            "width": img_info["width"],
            "height": img_info["height"],
            "annotations": annotations,
        }

    return image_records, category_id_to_name


image_records, category_id_to_name = build_image_records(valid_stems, stem_to_ann_paths)
print(f"이미지 {len(image_records)}장, 클래스 {len(category_id_to_name)}종")


## 도구 1 : annotation 클래스가 실제 이미지와 맞는지 확인하는 도구

두 단계로 확인한다.
1. **전수 자동 검증**: 폴더명에 박힌 진짜 약 코드(K-000123)와 annotation json의 dl_mapping_code를 전부 대조. 하나라도 안 맞으면 그 자리에서 바로 알 수 있다.
2. **육안 검수용 그리드**: 무작위로 뽑은 이미지에 박스 + 클래스명을 그려서 한 화면에 모아 보여준다. 멘토님이 예전 기수에서 쓰셨다는 것과 같은 방식(육안 확인)인데, 클래스 이름까지 같이 찍어줘서 "그림이랑 이름이 진짜 맞나"를 바로 확인할 수 있게 했다.

In [ ]:
#@title (1-A) 전수 자동 검증: 폴더코드 vs dl_mapping_code
FOLDER_CODE_RE = re.compile(r"(K-\d+)$")

n_checked = 0
mismatches = []

for stem, record in tqdm(image_records.items(), desc="검증 중"):
    for ann in record["annotations"]:
        ann_path = ann["annotation_path"]
        folder_name = os.path.basename(os.path.dirname(ann_path))
        m = FOLDER_CODE_RE.search(folder_name)
        if m is None:
            continue
        folder_code = m.group(1)

        with open(ann_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        dl_mapping_code = data["images"][0]["dl_mapping_code"]

        n_checked += 1
        if dl_mapping_code != folder_code:
            mismatches.append((ann_path, folder_code, dl_mapping_code))

print(f"검증한 annotation 수: {n_checked}")
print(f"불일치 건수: {len(mismatches)}")
if mismatches:
    print("\n불일치 샘플 (최대 10개):")
    for path, folder_code, dl_mapping_code in mismatches[:10]:
        print(f"  {path}\n    폴더코드={folder_code} vs dl_mapping_code={dl_mapping_code}")
else:
    print("전부 일치. dl_mapping_code를 클래스 식별자로 써도 안전하다는 뜻.")


In [ ]:
#@title (1-B) 육안 검수용 그리드 (박스 + 클래스명)
def draw_annotated(image, record, category_id_to_name, color=(255, 0, 0)):
    vis = image.copy()
    for ann in record["annotations"]:
        x, y, w, h = [int(round(v)) for v in ann["bbox"]]
        cv2.rectangle(vis, (x, y), (x + w, y + h), color, 3)
        name = category_id_to_name[ann["category_id"]]
        cv2.putText(vis, name, (x, max(y - 8, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return vis


def show_annotation_grid(stems, n_cols=3, title=""):
    n = len(stems)
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    axes = np.atleast_1d(axes).reshape(-1)

    for ax, stem in zip(axes, stems):
        record = image_records[stem]
        image = cv2.cvtColor(cv2.imread(os.path.join(TRAIN_IMAGE_DIR, record["file_name"])), cv2.COLOR_BGR2RGB)
        vis = draw_annotated(image, record, category_id_to_name)
        ax.imshow(vis)
        ax.set_title(stem, fontsize=8)
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


# 무작위로 9장 뽑아서 육안 검수 (재실행할 때마다 다른 9장이 나옴 - 여러 번 돌려보면서 확인)
sample_stems = random.sample(list(image_records.keys()), 9)
show_annotation_grid(sample_stems, n_cols=3, title="무작위 육안 검수 - 박스/클래스명이 사진과 맞는지 확인")


In [ ]:
#@title (1-C) 클래스별로 지정해서 검수하기 (특정 클래스가 의심스러울 때)
def show_class_samples(category_id, n=9):
    stems_with_class = [s for s, r in image_records.items()
                         if any(a["category_id"] == category_id for a in r["annotations"])]
    picked = random.sample(stems_with_class, min(n, len(stems_with_class)))
    show_annotation_grid(picked, n_cols=3, title=f"{category_id_to_name[category_id]} (category_id={category_id}) 검수")


# 사용 예시: 인스턴스가 적어서 라벨 오류 영향을 크게 받는 클래스부터 확인해보고 싶을 때
# show_class_samples(list(category_id_to_name.keys())[0])


In [ ]:
#@title (1-D) 중복/충돌 annotation 탐지 (같은 위치에 클래스 2개가 동시에 붙어있는 경우)
# 실제로 3,731장을 전수 조사해보니 30장에서, 같은 조합 폴더 안의 서로 다른 약 json 두 개가
# "완전히 같은 자리(IoU 0.5 이상)"를 서로 다른 category_id로 동시에 가리키고 있었다.
# 즉 한 알약 자리에 클래스 라벨이 두 개 붙어있는 셈이라, 이 상태로 학습하면 모델이
# 같은 자리에 대해 서로 다른 정답을 배우게 된다. 어느 쪽이 맞는지 자동으로는 판단이
# 안 되므로(둘 다 그 조합에 실제로 들어있는 약이긴 함), 기본값은 "둘 다 학습에서
# 제외"로 뒀다 - 틀린 라벨을 남기는 것보다 그 30장만 포기하는 게 안전하다고 판단.
def find_duplicate_annotations(image_records, iou_threshold=0.5):
    def iou_xywh(a, b):
        ax1, ay1, ax2, ay2 = a[0], a[1], a[0] + a[2], a[1] + a[3]
        bx1, by1, bx2, by2 = b[0], b[1], b[0] + b[2], b[1] + b[3]
        ix1, iy1 = max(ax1, bx1), max(ay1, by1)
        ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
        inter = iw * ih
        if inter == 0:
            return 0.0
        union = a[2] * a[3] + b[2] * b[3] - inter
        return inter / union

    conflicts = []  # (stem, idx_i, idx_j, iou)
    for stem, record in image_records.items():
        anns = record["annotations"]
        for i in range(len(anns)):
            for j in range(i + 1, len(anns)):
                iou = iou_xywh(anns[i]["bbox"], anns[j]["bbox"])
                if iou > iou_threshold and anns[i]["category_id"] != anns[j]["category_id"]:
                    conflicts.append((stem, i, j, iou))
    return conflicts


conflicts = find_duplicate_annotations(image_records)
print(f"클래스 충돌 annotation 쌍: {len(conflicts)}개 (전체 {len(image_records)}장 중 {len({c[0] for c in conflicts})}장에서 발견)")

for stem, i, j, iou in conflicts[:10]:
    r = image_records[stem]
    name_i = category_id_to_name[r["annotations"][i]["category_id"]]
    name_j = category_id_to_name[r["annotations"][j]["category_id"]]
    print(f"  {stem}: {name_i} vs {name_j} (IoU={iou:.2f})")


In [ ]:
#@title (1-E) 충돌 annotation 제외한 클린 버전 만들기
conflict_stems = {stem for stem, i, j, iou in conflicts}
conflict_indices = {}
for stem, i, j, iou in conflicts:
    conflict_indices.setdefault(stem, set()).update({i, j})

clean_image_records = {}
n_removed = 0
for stem, record in image_records.items():
    bad_idx = conflict_indices.get(stem, set())
    if not bad_idx:
        clean_image_records[stem] = record
        continue
    kept = [a for idx, a in enumerate(record["annotations"]) if idx not in bad_idx]
    n_removed += len(bad_idx)
    clean_image_records[stem] = {**record, "annotations": kept}

print(f"충돌이 있던 이미지: {len(conflict_stems)}장")
print(f"제외한 annotation 수: {n_removed}개")
print("이후 셀(박스 타이트닝 등)에서는 image_records 대신 clean_image_records를 쓰면 된다.")

# 충돌났던 이미지들을 직접 눈으로 보고 싶으면:
if conflict_stems:
    show_annotation_grid(list(conflict_stems)[:9], n_cols=3, title="클래스 충돌이 발견된 이미지들")


## 도구 2 : 박스 타이트닝 (루즈한 GT bbox를 알약 경계에 맞게 줄이기)

원본 annotation의 bbox가 알약보다 여유 있게(루즈하게) 잡혀 있어서, IoU 0.75 이상을 요구하는
채점 기준에서 손해를 본다는 문제였다. 배경이 단색에 가깝다는 점을 이용해서, 박스 안에서
"배경색과 얼마나 다른가"로 알약 영역만 골라내고 그 영역의 딱 맞는 사각형을 새 박스로 쓴다.

실제 샘플 21개로 테스트했을 때 평균적으로 원래 박스 면적의 78% 정도로 줄었고(즉 22% 정도가
불필요한 여백이었다는 뜻), 최소 38%~최대 100%(이미 타이트했던 경우 그대로 유지)로 나왔다.

In [ ]:
#@title (2-A) 박스 타이트닝 함수
def tighten_bbox(image, bbox, pad=15, margin=4, min_area_ratio=0.15, bg_sample=6):
    """image: BGR ndarray (cv2.imread 결과). bbox: [x, y, w, h].
    반환: 타이트닝된 [x, y, w, h]. 세그멘테이션이 애매하면 원본 bbox를 그대로 돌려준다
    (잘못 줄이는 것보다 안 줄이는 게 안전하다는 판단)."""
    H, W = image.shape[:2]
    x, y, w, h = bbox
    cx0, cy0 = x + w / 2, y + h / 2

    x0, y0 = max(0, int(x - pad)), max(0, int(y - pad))
    x1, y1 = min(W, int(x + w + pad)), min(H, int(y + h + pad))
    crop = image[y0:y1, x0:x1]
    if crop.size == 0:
        return bbox

    # 배경색 추정: 패딩을 넉넉히 준 크롭의 테두리 띠 (알약은 중앙에 있다고 가정)
    border_px = np.concatenate([
        crop[:bg_sample, :].reshape(-1, 3), crop[-bg_sample:, :].reshape(-1, 3),
        crop[:, :bg_sample].reshape(-1, 3), crop[:, -bg_sample:].reshape(-1, 3),
    ], axis=0)
    bg_color = np.median(border_px, axis=0)

    dist = np.linalg.norm(crop.astype(np.float32) - bg_color.astype(np.float32), axis=2)
    thresh = max(20.0, np.std(dist) * 0.5 + np.median(dist))
    mask = (dist > thresh).astype(np.uint8) * 255

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return bbox

    # 원래 박스 중심에 가장 가까우면서 충분히 큰 윤곽선을 고른다.
    # (옆 칸의 다른 알약이 패딩 영역에 살짝 걸려 들어오는 걸 막기 위해)
    def score(c):
        area = cv2.contourArea(c)
        if area < min_area_ratio * crop.shape[0] * crop.shape[1]:
            return -1
        bx, by, bw, bh = cv2.boundingRect(c)
        center_dist = np.hypot((x0 + bx + bw / 2) - cx0, (y0 + by + bh / 2) - cy0)
        return area / (1 + center_dist)

    best = max(contours, key=score)
    if score(best) <= 0:
        return bbox

    cx, cy, cw, ch = cv2.boundingRect(best)
    new_x = max(0, x0 + cx - margin)
    new_y = max(0, y0 + cy - margin)
    new_x2 = min(W, x0 + cx + cw + margin)
    new_y2 = min(H, y0 + cy + ch + margin)
    tight = [new_x, new_y, new_x2 - new_x, new_y2 - new_y]

    if (tight[2] * tight[3]) < min_area_ratio * (w * h):
        return bbox  # 너무 심하게 줄면 세그멘테이션 실패로 보고 원본 유지

    return tight


In [ ]:
#@title (2-B) 전체 데이터에 일괄 적용 + 통계 확인
tightened_records = {}
area_ratios = []
n_changed = 0

for stem, record in tqdm(clean_image_records.items(), desc="박스 타이트닝 중"):
    image = cv2.imread(os.path.join(TRAIN_IMAGE_DIR, record["file_name"]))
    new_annotations = []
    for ann in record["annotations"]:
        old_bbox = ann["bbox"]
        new_bbox = tighten_bbox(image, old_bbox)
        old_area = old_bbox[2] * old_bbox[3]
        new_area = new_bbox[2] * new_bbox[3]
        area_ratios.append(new_area / old_area if old_area else 1.0)
        if new_bbox != old_bbox:
            n_changed += 1
        new_annotations.append({**ann, "bbox": new_bbox, "original_bbox": old_bbox})

    tightened_records[stem] = {**record, "annotations": new_annotations}

area_ratios = np.array(area_ratios)
print(f"전체 박스 수: {len(area_ratios)}")
print(f"실제로 줄어든 박스 수: {n_changed} ({n_changed/len(area_ratios):.1%})")
print(f"면적 비율(타이트닝후/원본) 평균 {area_ratios.mean():.1%} | "
      f"중앙값 {np.median(area_ratios):.1%} | 최소 {area_ratios.min():.1%} | 최대 {area_ratios.max():.1%}")

plt.figure(figsize=(8, 4))
plt.hist(area_ratios, bins=30)
plt.xlabel("타이트닝 후 면적 / 원본 면적")
plt.ylabel("박스 개수")
plt.title("박스 타이트닝 전후 면적 비율 분포")
plt.tight_layout()
plt.show()


In [ ]:
#@title (2-C) Before/After 그리드로 육안 확인
def show_tighten_grid(stems, n_cols=3):
    n = len(stems)
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    axes = np.atleast_1d(axes).reshape(-1)

    for ax, stem in zip(axes, stems):
        record = tightened_records[stem]
        image = cv2.cvtColor(cv2.imread(os.path.join(TRAIN_IMAGE_DIR, record["file_name"])), cv2.COLOR_BGR2RGB)
        vis = image.copy()
        for ann in record["annotations"]:
            ox, oy, ow, oh = [int(v) for v in ann["original_bbox"]]
            cv2.rectangle(vis, (ox, oy), (ox + ow, oy + oh), (255, 0, 0), 3)  # 빨강 = 원본
            nx, ny, nw, nh = [int(v) for v in ann["bbox"]]
            cv2.rectangle(vis, (nx, ny), (nx + nw, ny + nh), (0, 255, 0), 3)  # 초록 = 타이트닝 후
        ax.imshow(vis)
        ax.set_title(stem, fontsize=8)
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# 면적이 많이 줄어든(=원래 많이 루즈했던) 순서로 9장을 뽑아서 확인 - 가장 극적으로
# 바뀐 경우들을 봐야 알고리즘이 실수로 알약을 잘라먹진 않았는지 확인하기 좋다.
stem_worst_ratio = {}
for stem, record in tightened_records.items():
    ratios_here = [
        (a["bbox"][2] * a["bbox"][3]) / (a["original_bbox"][2] * a["original_bbox"][3])
        for a in record["annotations"] if a["original_bbox"][2] * a["original_bbox"][3] > 0
    ]
    if ratios_here:
        stem_worst_ratio[stem] = min(ratios_here)

worst_stems = sorted(stem_worst_ratio, key=stem_worst_ratio.get)[:9]
show_tighten_grid(worst_stems)


In [ ]:
#@title (2-D) 타이트닝된 annotation 저장 (기존 학습 노트북에서 바로 불러쓸 수 있게)
output_path = os.path.join(DATA_ROOT, "tightened_annotations.json")

serializable = {
    stem: {
        "file_name": record["file_name"],
        "width": record["width"],
        "height": record["height"],
        "annotations": [
            {"bbox": a["bbox"], "category_id": a["category_id"]}
            for a in record["annotations"]
        ],
    }
    for stem, record in tightened_records.items()
}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(serializable, f, ensure_ascii=False)

print(f"저장 완료: {output_path}")
print("각 학습 노트북의 build_image_records 결과(image_records)를 이걸로 그대로 바꿔 끼우면 됨:")
print('  image_records = json.load(open(".../tightened_annotations.json"))')


In [ ]:
#@title (2-E) wandb에 결과 기록 + tightened_annotations.json 아티팩트 업로드
# 다른 노트북들처럼 이 데이터 정제 작업도 wandb에 남겨서, 나중에 "이 라벨/박스가
# 언제 어떻게 정제된 버전인지" 팀이 같이 추적할 수 있게 한다. 학습 run이 아니라
# job_type="preprocess"로 구분해서 기록.
import wandb

run = wandb.init(
    entity="jaedong0817--org",
    project="beginner-team-project",
    job_type="preprocess",
    name="annotation-cleanup-and-bbox-tighten",
    config={
        "iou_threshold_for_conflict": 0.5,
        "tighten_pad": 15,
        "tighten_margin": 4,
        "tighten_min_area_ratio": 0.15,
    },
)

run.summary["n_images_total"] = len(image_records)
run.summary["n_class_conflicts"] = len(conflicts)
run.summary["n_images_with_conflict"] = len({c[0] for c in conflicts})
run.summary["bbox_area_ratio_mean"] = float(area_ratios.mean())
run.summary["bbox_area_ratio_median"] = float(np.median(area_ratios))
run.summary["n_boxes_changed"] = int(n_changed)

artifact = wandb.Artifact(
    name="tightened-annotations",
    type="dataset",
    description="dl_mapping_code 기준 재매핑 + 클래스 충돌 제거 + 박스 타이트닝까지 끝낸 annotation",
)
artifact.add_file(output_path)
run.log_artifact(artifact)

run.finish()
print("wandb 기록 완료. 팀 프로젝트 페이지에서 annotation-cleanup-and-bbox-tighten run으로 확인 가능.")
